# Health Score EDA

## Purpose
Train a Health Score model using **flowchart factors only** (per Mermaid):
- **F_Utilization** (calculated_utilization): from `licenses_total`, `licenses_used`, `utilization_percentage`
- **M_Rel** (Relationship Score): `relationship_score`

**Target:** `health_score` (continuous).

## Flowchart inputs
| Input | Source |
|-------|--------|
| calculated_utilization | licenses_used / licenses_total or utilization_percentage/100 |
| relationship_score | From Relationship model / column |


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import joblib
import json

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import RidgeCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries OK')

Libraries OK


## 1. Load data

In [2]:
# Resolve path to Research/customer_data_25000.xlsx
cwd = Path.cwd()
research = cwd if (cwd / 'customer_data_25000.xlsx').exists() else (cwd.parent if (cwd.parent / 'customer_data_25000.xlsx').exists() else cwd / 'Research')
if not (research / 'customer_data_25000.xlsx').exists():
    research = Path(r'D:\Projects_Main\Renewal-Upsell-Advisor\Research')
data_path = research / 'customer_data_25000.xlsx'
df = pd.read_excel(data_path, sheet_name='Accounts')
print(f'Loaded: {df.shape[0]:,} rows, {df.shape[1]} cols')
df.head()

Loaded: 25,000 rows, 27 cols


,name,domain,industry,company_size,arr,mrr,contract_start_date,contract_end_date,renewal_date,last_contact_date,...,sentiment_category,licenses_total,licenses_used,utilization_percentage,csm_name,csm_email,primary_contact_name,primary_contact_email,primary_contact_phone,salesforce_id
0,Cole LLC,colellc.com,Technology,Medium,156049,13004.08,2025-06-12,2026-06-12,2026-06-12,2026-02-14 10:30:11,...,negative,20,15,79,Emily Rodriguez,emily.rodriguez@company.com,Danielle Johnson,john21@example.net,001-581-896-0013x3890,SF-835392
1,"Stevens, Martinez and Nielsen",stevensmartinezandnielsen.com,Media,Enterprise,378618,31551.50,2025-05-31,2026-05-31,2026-05-31,2026-02-13 10:30:11,...,very_negative,30,23,79,Sarah Chen,sarah.chen@company.com,Lisa Smith,helenpeterson@example.org,651.216.1559,NaN
2,Clark-Adams,clark-adams.com,Manufacturing,Medium,62823,5235.25,2025-05-06,2026-05-06,2026-05-06,2026-02-07 10:30:11,...,very_positive,150,52,35,Emily Rodriguez,emily.rodriguez@company.com,Christian Carter,barbara10@example.net,441.731.6475,SF-148050
3,"Porter, Wilkerson and Day",porterwilkersonandday.com,Research,Medium,96054,8004.50,2025-06-15,2026-06-15,2026-06-15,2026-02-14 10:30:11,...,very_negative,200,142,71,Sarah Chen,sarah.chen@company.com,Sharon Wong,amandasanchez@example.com,(748)535-0305x6413,SF-356702
4,Carlson-Mcdonald,carlson-mcdonald.com,Consulting,Large,257691,21474.25,2025-04-12,2026-04-12,2026-04-12,2026-01-23 10:30:11,...,very_positive,100,32,32,James Wilson,james.wilson@company.com,Douglas Taylor,julie69@example.com,(332)887-1012x269,NaN


## 2. Flowchart features: F_Utilization + relationship_score

**F_Utilization (calculated_utilization):** Use utilization_percentage/100, or licenses_used/licenses_total when total > 0.

In [3]:
X = df.copy()
# F_Utilization: single metric 0-1 (flowchart)
X['calculated_utilization'] = np.where(
    X['licenses_total'] > 0,
    X['licenses_used'].clip(0, X['licenses_total']) / X['licenses_total'],
    X['utilization_percentage'].fillna(50) / 100.0
)
# M_Rel: Relationship Score (flowchart)
if 'relationship_score' not in X.columns:
    raise ValueError('relationship_score required for Health model')

feature_cols = ['calculated_utilization', 'relationship_score']
X = X[feature_cols].copy()
X = X.fillna(X.median())

y = df['health_score'].copy()
y = y.fillna(y.median())

print('Features (flowchart only):', feature_cols)
print('Target: health_score')
X.head()

Features (flowchart only): ['calculated_utilization', 'relationship_score']
Target: health_score


,calculated_utilization,relationship_score
0,0.750000,65.65
1,0.766667,54.40
2,0.346667,64.05
3,0.710000,53.12
4,0.320000,54.68


## 3. Train/test split and scale

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print('Train:', X_train_s.shape, 'Test:', X_test_s.shape)

Train: (20000, 2) Test: (5000, 2)


## 4a. Train each model with hyperparameter tuning

Train all base models with `RandomizedSearchCV`; then stack them in the next section.

In [5]:
# Train each model with hyperparameter tuning; store best estimators in `models`
models = {}
results = {}
best_params = {}

# 1. Ridge (already has built-in CV for alpha)
print("1. Ridge (RidgeCV)...")
ridge = RidgeCV(alphas=[0.1, 1.0, 10.0]).fit(X_train_s, y_train)
models['Ridge'] = ridge
pred = ridge.predict(X_test_s)
results['Ridge'] = {'r2': r2_score(y_test, pred), 'mae': mean_absolute_error(y_test, pred)}
print(f"   R²={results['Ridge']['r2']:.4f}, MAE={results['Ridge']['mae']:.4f}")

# 2. XGBoost
print("2. XGBoost (RandomizedSearchCV)...")
xgb_base = XGBRegressor(random_state=42, n_jobs=-1)
xgb_params = {'n_estimators': [150, 200, 300], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5, 7], 'subsample': [0.8, 0.9], 'reg_alpha': [0.1, 0.5], 'reg_lambda': [1.0, 2.0]}
xgb_search = RandomizedSearchCV(xgb_base, xgb_params, n_iter=20, cv=5, scoring='r2', n_jobs=-1, random_state=42)
xgb_search.fit(X_train_s, y_train)
models['XGBoost'] = xgb_search.best_estimator_
best_params['XGBoost'] = xgb_search.best_params_
pred = models['XGBoost'].predict(X_test_s)
results['XGBoost'] = {'r2': r2_score(y_test, pred), 'mae': mean_absolute_error(y_test, pred)}
print(f"   Best params: {xgb_search.best_params_}\n   R²={results['XGBoost']['r2']:.4f}, MAE={results['XGBoost']['mae']:.4f}")

# 3. LightGBM
print("3. LightGBM (RandomizedSearchCV)...")
lgb_base = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
lgb_params = {'n_estimators': [150, 200, 300], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5, 7], 'num_leaves': [31, 50], 'subsample': [0.8, 0.9], 'reg_alpha': [0.1, 0.5], 'reg_lambda': [1.0, 2.0]}
lgb_search = RandomizedSearchCV(lgb_base, lgb_params, n_iter=20, cv=5, scoring='r2', n_jobs=-1, random_state=42)
lgb_search.fit(X_train_s, y_train)
models['LightGBM'] = lgb_search.best_estimator_
best_params['LightGBM'] = lgb_search.best_params_
pred = models['LightGBM'].predict(X_test_s)
results['LightGBM'] = {'r2': r2_score(y_test, pred), 'mae': mean_absolute_error(y_test, pred)}
print(f"   Best params: {lgb_search.best_params_}\n   R²={results['LightGBM']['r2']:.4f}, MAE={results['LightGBM']['mae']:.4f}")

# 4. CatBoost
print("4. CatBoost (RandomizedSearchCV)...")
cat_base = CatBoostRegressor(random_state=42, verbose=False)
cat_params = {'iterations': [150, 200, 300], 'learning_rate': [0.05, 0.1], 'depth': [3, 5, 7], 'subsample': [0.8, 0.9], 'reg_lambda': [1.0, 2.0]}
cat_search = RandomizedSearchCV(cat_base, cat_params, n_iter=15, cv=5, scoring='r2', n_jobs=-1, random_state=42)
cat_search.fit(X_train_s, y_train)
models['CatBoost'] = cat_search.best_estimator_
best_params['CatBoost'] = cat_search.best_params_
pred = models['CatBoost'].predict(X_test_s)
results['CatBoost'] = {'r2': r2_score(y_test, pred), 'mae': mean_absolute_error(y_test, pred)}
print(f"   Best params: {cat_search.best_params_}\n   R²={results['CatBoost']['r2']:.4f}, MAE={results['CatBoost']['mae']:.4f}")

# 5. Random Forest
print("5. Random Forest (RandomizedSearchCV)...")
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_params = {'n_estimators': [150, 200, 300], 'max_depth': [5, 8, 10, None], 'min_samples_split': [2, 5], 'min_samples_leaf': [1, 2], 'max_features': ['sqrt', 'log2']}
rf_search = RandomizedSearchCV(rf_base, rf_params, n_iter=15, cv=5, scoring='r2', n_jobs=-1, random_state=42)
rf_search.fit(X_train_s, y_train)
models['RandomForest'] = rf_search.best_estimator_
best_params['RandomForest'] = rf_search.best_params_
pred = models['RandomForest'].predict(X_test_s)
results['RandomForest'] = {'r2': r2_score(y_test, pred), 'mae': mean_absolute_error(y_test, pred)}
print(f"   Best params: {rf_search.best_params_}\n   R²={results['RandomForest']['r2']:.4f}, MAE={results['RandomForest']['mae']:.4f}")

print("\n--- Model comparison (after hyperparameter tuning) ---")
for name, m in results.items():
    print(f"  {name}: R²={m['r2']:.4f}, MAE={m['mae']:.4f}")

1. Ridge (RidgeCV)...
   R²=0.9474, MAE=3.1241
2. XGBoost (RandomizedSearchCV)...
   Best params: {'subsample': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05}
   R²=0.9470, MAE=3.1339
3. LightGBM (RandomizedSearchCV)...
   Best params: {'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.1, 'num_leaves': 50, 'n_estimators': 150, 'max_depth': 3, 'learning_rate': 0.05}
   R²=0.9470, MAE=3.1366
4. CatBoost (RandomizedSearchCV)...
   Best params: {'subsample': 0.8, 'reg_lambda': 2.0, 'learning_rate': 0.05, 'iterations': 150, 'depth': 7}
   R²=0.9469, MAE=3.1391
5. Random Forest (RandomizedSearchCV)...
   Best params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 10}
   R²=0.9455, MAE=3.1876

--- Model comparison (after hyperparameter tuning) ---
  Ridge: R²=0.9474, MAE=3.1241
  XGBoost: R²=0.9470, MAE=3.1339
  LightGBM: R²=0.9470, MAE=3.1366
  CatBoost: R²=0.9469, MAE=3.1391
 

## 4b. Stack tuned models

Stack the hyperparameter-tuned base models (XGBoost, LightGBM, CatBoost) with a Ridge meta-learner.

In [6]:
# Stack tuned models (separate step)
stacker = StackingRegressor(
    estimators=[('xgb', models['XGBoost']), ('lgb', models['LightGBM']), ('cat', models['CatBoost'])],
    final_estimator=RidgeCV(alphas=[0.1, 1.0, 10.0]), cv=5
)
stacker.fit(X_train_s, y_train)
pred_s = stacker.predict(X_test_s)
print(f"Stacking: R²={r2_score(y_test, pred_s):.4f}, MAE={mean_absolute_error(y_test, pred_s):.4f}")

Stacking: R²=0.9471, MAE=3.1316


## 5. Save model artifacts (optional)

In [7]:
# Save all trained base models + best model (stacker)
save_dir = Path(r'D:\Projects_Main\Renewal-Upsell-Advisor\Research\HealthScore\models')
save_dir.mkdir(parents=True, exist_ok=True)

# Save all tuned base models
name_to_file = {'Ridge': 'health_ridge', 'XGBoost': 'health_xgb', 'LightGBM': 'health_lgb', 'CatBoost': 'health_cat', 'RandomForest': 'health_rf'}
for name, model in models.items():
    fname = name_to_file.get(name, name.lower().replace(' ', '_'))
    joblib.dump(model, save_dir / f"{fname}.joblib")
    print(f"✓ Saved: {name} -> {fname}.joblib")

# Save best model (stacking ensemble)
joblib.dump(stacker, save_dir / 'health_stacker_best.joblib')
print("✓ Saved: Best model (Stacking) -> health_stacker_best.joblib")

# Save best hyperparameters (for base models that were tuned)
if best_params:
    joblib.dump(best_params, save_dir / 'health_best_params.joblib')
    print("✓ Saved: Best hyperparameters -> health_best_params.joblib")

joblib.dump(scaler, save_dir / 'health_scaler.joblib')
with open(save_dir / 'health_feature_names.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)
print("✓ Saved: Scaler and feature names")
print("Location:", save_dir)

✓ Saved: Ridge -> health_ridge.joblib
✓ Saved: XGBoost -> health_xgb.joblib
✓ Saved: LightGBM -> health_lgb.joblib
✓ Saved: CatBoost -> health_cat.joblib
✓ Saved: RandomForest -> health_rf.joblib
✓ Saved: Best model (Stacking) -> health_stacker_best.joblib
✓ Saved: Best hyperparameters -> health_best_params.joblib
✓ Saved: Scaler and feature names
Location: D:\Projects_Main\Renewal-Upsell-Advisor\Research\HealthScore\models
